## Scenariusz symulacji

Scenariusz przedstawia statyczną analizę propagacji sygnału radiowego w modelu sceny 3D kampusu AGH. W scenariuszu rozpatrywany jest jeden nadajniki oraz kilka odbiorników w różnych miejscach symulujących telefony komórkowe osób. Wszystkie obiekty środowiska pozostają nieruchome (układ statyczny). Celem jest pokazanie różnorodności propagacji poprzez analizę ścieżek LOS, ścieżek odbitych oraz ścieżek przenikających przez przeszkody (np. budynki). W kolejnym etapie prezentowana jest mapa pokrycia nadajnika, aby zwizualizować lokalne zaniki sygnału powodowane przez obiekty takie jak człowiek, samochód i budynek.

In [1]:
# Importowanie najwazniejszych bibliotek
import numpy as np
from IPython.display import clear_output

# Importowanie najwazniejszych komponentow Sionna RT
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver, RadioMapSolver, subcarrier_frequencies, RadioMaterial

In [2]:
scene = load_scene("./OpenStreetMap/kampus_AGH_B9_test/Kampus_AGH_B9_test.xml", merge_shapes=False)

# Tworzenie materiału udającego próżnię
vacuum = RadioMaterial("vacuum", relative_permittivity=1.0, conductivity=0.0)

scene.add(vacuum)

for name, obj in scene.objects.items():
    if "Czlowiek" in name:
        obj.radio_material = vacuum

# konfiguracja parametrów anteny dla nadajnika
scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="tr38901",
                             polarization="V")

# konfiguracja parametrów anteny dla odbiornika
scene.rx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="dipole",
                             polarization="cross")

# Tworzenie nadajnika
tx = Transmitter(name="tx",
                 position=[-9.169, 87.045, 17.500],
                 display_radius=3)

# dodanie instancji nadajnika do sceny
scene.add(tx)

# utworzenie odbiornika
rx_czlowiek_1 = Receiver(name="rx1",
              position=[4.08414, 76.9666, 1.500],
              display_radius=3)

scene.add(rx_czlowiek_1)

rx_czlowiek_2 = Receiver(name="rx2",
              position=[-9.94801, 62.2862, 1.500],
              display_radius=3)

scene.add(rx_czlowiek_2)

rx_czlowiek_3 = Receiver(name="rx3",
              position=[2.12167, 27.8497, 1.500],
              display_radius=3)

scene.add(rx_czlowiek_3)

rx_czlowiek_4 = Receiver(name="rx4",
              position=[-53.6909, 31.1754, 1.500],
              display_radius=3)

scene.add(rx_czlowiek_4)

rx_czlowiek_5 = Receiver(name="rx5",
              position=[-20.1955, 76.4766, 1.500],
              display_radius=3)

scene.add(rx_czlowiek_5)

rx_czlowiek_6 = Receiver(name="rx6",
              position=[40.5994, 100.325, 1.500],
              display_radius=3)

scene.add(rx_czlowiek_6)

tx.look_at(rx_czlowiek_1) # nastawienie orientacji nadajnika w kierunku odbiornika

p_solver  = PathSolver()

# Wyznaczenie ścieżek propagacji sygnału między nadajnikiem a odbiornikiem z uwzględnieniem różnych zjawisk propagacyjnych.
paths = p_solver(scene=scene,
                 max_depth=5,
                 samples_per_src=5_000_000,
                 max_num_paths_per_src=1000,
                 los=True,
                 specular_reflection=True,
                 diffuse_reflection=True,
                 refraction=True,
                 synthetic_array=False,
                 seed=41)


p = scene.preview(paths=paths, resolution=(800, 600), fov=60); 

In [4]:
# Czyszczenie rx dla leszej wizualizacji
for rx_name in list(scene.receivers.keys()):
    if "rx" in rx_name:
        scene.remove(rx_name)

# Ustawienie odpowieniego materiału dla człowieka
human_flesh = RadioMaterial("human_flesh", relative_permittivity=45.0, conductivity=1.5, color=[1.0, 0.4, 0.8])

if "human_flesh" not in scene.radio_materials:
    scene.add(human_flesh)

    for name, obj in scene.objects.items():
        if "Czlowiek" in name:
            obj.radio_material = human_flesh

# Pobieramy granice (bounding box) całej naszej sceny
bbox = scene.mi_scene.bbox()
bbox_min = np.array(bbox.min)
bbox_max = np.array(bbox.max)

# Obliczamy szerokość (X) i długość (Y) mapy (wymuszamy typ float)
size_x = float(bbox_max[0] - bbox_min[0])
size_y = float(bbox_max[1] - bbox_min[1])
map_size = [size_x, size_y]

# Obliczamy środek kampusu i znowu wymuszamy typ float dla X i Y
center_x = float((bbox_max[0] + bbox_min[0]) / 2)
center_y = float((bbox_max[1] + bbox_min[1]) / 2)
map_center = [center_x, center_y, 0.5]

# Orientacja mapy w przestrzeni (brak obrotu na osiach X, Y, Z)
map_orientation = [0, 0, 0]

rm_solver = RadioMapSolver()

rm = rm_solver(scene=scene,
               max_depth=3,
               cell_size=[0.2,0.2],
               samples_per_tx=10**6,
               center=map_center,
               size=map_size,
               orientation=map_orientation)

scene.preview(radio_map=rm)